# Frequency-Aware Latent Graph Diffusion (FALD) — Colab Workspace

**Run this notebook in a GPU runtime** (Runtime > Change runtime type > T4/A100).

This notebook is the single entry point for running the entire project on Google Colab.
It clones the repo, installs all dependencies (DiGress, PyG, ORCA, graph-tool),
and provides runnable cells for every stage from the `PLAN.md`.

### Current status (as of the last Windows session)

| Stage | Status |
|---|---|
| 1. Environment & DiGress | ✅ |
| 2. Evaluation harness | ✅ |
| 3. Spectral utilities & SignNet | ✅ |
| 3.5. Adjacency diffusion baseline | ✅ |
| 4. Autoencoder investigation | ✅ (decision: adjacency path) |
| 5. Unconditional adjacency diffusion | ✅ |
| 6. Kill gate (oracle conditioning) | ✅ PASSED |
| 6.5. Discrete DiGress validity repair | ⚠ red on validity |
| 7–10 | Not started |

**What this notebook provides:** everything needed to continue from Stage 6.5 onward,
plus the ability to re-run any earlier stage to verify results on Colab hardware.

---
## 0. Environment Setup

Run these cells once per Colab session. They are idempotent.

In [ ]:
# Check GPU availability
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

In [ ]:
import os

REPO_URL = "https://github.com/TamaraBluzer/Frequency_aware_diffusion-.git"
REPO_DIR = "/content/FALD"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only || echo "Pull failed (maybe local changes); using existing checkout."

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Run the full setup: DiGress clone+patch, ORCA build, all pip installs.
# Takes ~2-3 minutes on first run, <30s on re-run.
!bash scripts/colab_setup.sh

In [ ]:
# Verify imports work
import torch
print(f"PyTorch {torch.__version__}, CUDA {torch.version.cuda}, GPU: {torch.cuda.get_device_name(0)}")

import networkx as nx
import numpy as np
from fald.data import load_splits, build_condition_tensors
from fald.eval import GraphEvaluator
from fald.eval.orca import is_available as orca_available
from fald.models import AdjacencyDiffusion, AdjacencyDiffusionConfig, DiscreteAdjacencyDiffusion
from fald.paths import repo_root, work_dir, data_dir, third_party_dir, checkpoints_dir, results_dir

print(f"ORCA available: {orca_available()}")
print(f"Repo root: {repo_root()}")
print(f"Work dir: {work_dir()}")
print("All imports OK.")

### Optional: Mount Google Drive for persistent checkpoints

Colab VMs are ephemeral — files are lost when the runtime disconnects.
Mount Google Drive to persist checkpoints and results across sessions.

In [ ]:
# Uncomment these lines to mount Google Drive and use it for heavy artifacts.
# Once mounted, FALD_WORK_DIR redirects checkpoints/data/third_party there.

# from google.colab import drive
# drive.mount('/content/drive')
#
# DRIVE_WORK_DIR = "/content/drive/MyDrive/FALD"
# os.makedirs(DRIVE_WORK_DIR, exist_ok=True)
# os.environ["FALD_WORK_DIR"] = DRIVE_WORK_DIR
# print(f"FALD_WORK_DIR set to {DRIVE_WORK_DIR}")
# print("Checkpoints and data will persist across Colab sessions.")

---
## 1. Stage 1 — DiGress Smoke Test

Already passed. Re-run to verify Colab environment works end-to-end.

In [ ]:
# Quick ConGress smoke test (5 epochs, tiny model — not a real training run)
%cd {os.path.join(third_party_dir(), 'digress', 'src')}
!MPLBACKEND=Agg python main.py dataset=planar model=continuous general.name=stage1_smoke \
    general.wandb=disabled general.gpus=1 \
    train.batch_size=8 train.n_epochs=2 \
    general.number_chain_steps=10 model.diffusion_steps=50 model.n_layers=2
%cd {REPO_DIR}

---
## 2. Stage 2 — Evaluation Harness Calibration

Verifies that the MMD evaluator is working correctly: train-vs-train near zero,
train-vs-ER large.

In [ ]:
!python scripts/calibrate_eval.py --dataset planar

---
## 3. Stage 3 — Spectral Sanity Checks

Verifies eigendecomposition, band selection, SignNet invariance, and the component-count offset.

In [ ]:
!python scripts/spectral_sanity.py

---
## 4. Stage 5 — Unconditional Adjacency Diffusion (`none` baseline)

Trains the continuous adjacency diffusion model with no spectral condition.
This is the baseline that Stage 6's conditioned models must beat.

In [ ]:
# Full training run — ~15-30 min on T4 depending on batch size.
# Reduce --epochs for a quick check; the model was trained at 200 epochs.
!python scripts/train_adjacency_diffusion.py \
    --dataset planar \
    --band none \
    --k 8 \
    --epochs 200 \
    --batch-size 16 \
    --seed 0

In [ ]:
# Seeds 1 and 2 for error bars
!python scripts/train_adjacency_diffusion.py --dataset planar --band none --k 8 --epochs 200 --seed 1
!python scripts/train_adjacency_diffusion.py --dataset planar --band none --k 8 --epochs 200 --seed 2

---
## 5. Stage 6 — Kill Gate: Oracle Spectral Conditioning

The central hypothesis test. `low, k=8` with oracle spectra must beat `none`.

**Previous result (Windows):** Ratio: `low` 157.97 ± 16.20 vs `none` 327.23 ± 8.80.
Kill gate PASSED.

In [ ]:
# Train low-band conditioned model (3 seeds)
for seed in [0, 1, 2]:
    !python scripts/train_adjacency_diffusion.py \
        --dataset planar --band low --k 8 --epochs 200 --seed {seed}

In [ ]:
# Train high-band and random-band controls (3 seeds each)
for band in ["high", "random"]:
    for seed in [0, 1, 2]:
        !python scripts/train_adjacency_diffusion.py \
            --dataset planar --band {band} --k 8 --epochs 200 --seed {seed}

In [ ]:
# Condition sensitivity check (T9 from PLAN.md)
!python scripts/check_condition_sensitivity.py

In [ ]:
# Summarize the pilot results with bootstrap CIs
!python scripts/summarize_adjacency_pilot.py

---
## 6. Stage 6.5 — Discrete Diffusion Validity Repair

The continuous model has 0% Planar validity (consistent with published ConGress).
Try discrete D3PM to recover structural validity while keeping the conditioning advantage.

**Previous result:** 0% validity after 500 epochs — needs more capacity/steps to match
published DiGress (10 layers, 1000 steps, all auxiliary features).

In [ ]:
# Discrete diffusion — larger model to attempt DiGress-level validity.
# This is the active frontier. Adjust n-layers, timesteps, and epochs as needed.
!python scripts/train_adjacency_diffusion.py \
    --dataset planar \
    --process discrete \
    --band none \
    --k 8 \
    --epochs 500 \
    --n-layers 6 \
    --timesteps 200 \
    --cycle-features \
    --allow-gate-failure \
    --seed 0

In [ ]:
# Discrete with low-band conditioning
!python scripts/train_adjacency_diffusion.py \
    --dataset planar \
    --process discrete \
    --band low \
    --k 8 \
    --epochs 500 \
    --n-layers 6 \
    --timesteps 200 \
    --cycle-features \
    --allow-gate-failure \
    --seed 0

### Scaling up toward published DiGress

Published DiGress uses 10 layers, 1000 timesteps, and all auxiliary features.
Colab T4 (16GB) can handle this; A100 is faster.

In [ ]:
# Attempt closer to published DiGress scale
!python scripts/train_adjacency_diffusion.py \
    --dataset planar \
    --process discrete \
    --band none \
    --k 8 \
    --epochs 1000 \
    --n-layers 10 \
    --timesteps 500 \
    --cycle-features \
    --allow-gate-failure \
    --seed 0

---
## 7. Stage 7 — The Frequency Sweep (Headline Result)

Six arms × k ∈ {2, 4, 8, 16, 32} on Planar, 3 seeds each.
Still oracle-conditioned so the frequency question is isolated.

**Note:** Run this on the process type that has non-zero validity,
or run it on continuous to report Ratio (with the validity caveat noted).

In [ ]:
# Full frequency sweep — continuous model (reports Ratio, not V.U.N.)
# This is a large grid; consider running overnight on Colab Pro.
import itertools

bands = ["none", "low", "high", "random"]
ks = [2, 4, 8, 16, 32]
seeds = [0, 1, 2]

for band, k, seed in itertools.product(bands, ks, seeds):
    if band == "none" and k != 8:
        continue  # none is the same regardless of k
    print(f"\n{'='*60}")
    print(f"  band={band}  k={k}  seed={seed}")
    print(f"{'='*60}")
    !python scripts/train_adjacency_diffusion.py \
        --dataset planar --band {band} --k {k} --epochs 200 --seed {seed}

In [ ]:
# Reduced SBM grid: low, high, none at k ∈ {2, 8, 32}
sbm_bands = ["none", "low", "high"]
sbm_ks = [2, 8, 32]

for band, k, seed in itertools.product(sbm_bands, sbm_ks, seeds):
    if band == "none" and k != 8:
        continue
    print(f"\n{'='*60}")
    print(f"  SBM: band={band}  k={k}  seed={seed}")
    print(f"{'='*60}")
    !python scripts/train_adjacency_diffusion.py \
        --dataset sbm --band {band} --k {k} --epochs 200 --seed {seed}

---
## 8. Stage 8 — Learned Spectral Prior

Small diffusion model over `(n, λ_k, U_k)` so we can sample end-to-end
without oracle inputs. This stage is not yet implemented — add cells here
as the model is built.

In [ ]:
# TODO: Stage 8 implementation
print("Stage 8 not yet implemented. See PLAN.md for specification.")

---
## 9. Stage 9 — Downstream Experiment & Controls

- SBM community-count classification augmentation study
- Ablations: cluster-conditioning, Laplacian variant, SignNet
- Memorization audit

In [ ]:
# TODO: Stage 9 implementation
print("Stage 9 not yet implemented. See PLAN.md for specification.")

---
## 10. Stage 10 — Report Figures

Regenerate all figures from the `results/` directory.

In [ ]:
# TODO: Figure generation script
print("Stage 10 not yet implemented. See PLAN.md for specification.")

---
## Utilities

### Run tests

In [ ]:
!python -m pytest tests/ -v --tb=short

### Inspect results

In [ ]:
import json
from pathlib import Path

results = sorted(Path("results").glob("*.json"))
print(f"Found {len(results)} result files:\n")
for path in results:
    with open(path) as f:
        data = json.load(f)
    band = data.get("band", "?")
    k = data.get("k", "?")
    seed = data.get("seed", "?")
    ratio = data.get("evaluation", {}).get("ratio", "N/A") if data.get("evaluation") else "N/A"
    gate = data.get("gate_passed", "N/A")
    print(f"  {path.name:55s}  band={band:6s} k={k:>2} seed={seed}  Ratio={ratio!s:>8s}  gate={gate}")

### Save results to Google Drive

In [ ]:
# Uncomment to copy results to Drive before the runtime disconnects.
# Make sure you mounted Drive in section 0 above.

# import shutil
# drive_results = "/content/drive/MyDrive/FALD/results"
# os.makedirs(drive_results, exist_ok=True)
# for f in Path("results").glob("*.json"):
#     shutil.copy2(f, drive_results)
#     print(f"Copied {f.name}")
# print(f"\nResults saved to {drive_results}")

### Commit and push results back to GitHub

In [ ]:
# Uncomment and fill in your token to push results from Colab.
# Use a fine-grained personal access token with Contents:write scope.

# GITHUB_TOKEN = "ghp_..."  # paste your token here
# !git remote set-url origin https://{GITHUB_TOKEN}@github.com/TamaraBluzer/Frequency_aware_diffusion-.git
# !git add results/*.json results/figures/
# !git commit -m "Add Colab results" || echo "Nothing to commit."
# !git push